# 05 - RF Tuned Current Stress
Template konsisten + output lokal + registry MLflow.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import sys
import json
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "nostressia-machine-learning" / "Current-Stress" / "notebooks" / "experiments",
]
for _dir in CANDIDATE_DIRS:
    if (_dir / "mlflow_utils.py").exists():
        sys.path.insert(0, str(_dir))
        break

from mlflow_utils import (
    RANDOM_STATE,
    configure_mlflow,
    set_seeds,
    load_current_stress_dataset,
    get_dataset_path,
    select_feature_set,
    split_data,
    build_preprocessor,
    evaluate_classification,
    log_classification_artifacts,
    log_run_metadata,
    create_local_output_dir,
)

repo_root = configure_mlflow()
set_seeds(RANDOM_STATE)


In [ ]:
NOTEBOOK_NAME = "05_rf_tuned_current_stress.ipynb"
RUN_NAME = "RF Tuned - Current Stress"
FEATURE_SET = "all"
REGISTERED_MODEL_NAME = "CurrentStress_RF_Tuned"

raw_df, feature_df, y = load_current_stress_dataset(repo_root)
dataset_source = str(get_dataset_path(repo_root))
X = select_feature_set(feature_df, FEATURE_SET)
X_train, X_test, y_train, y_test = split_data(X, y)

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()
preprocessor = build_preprocessor(num_cols, cat_cols)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

with mlflow.start_run(run_name=RUN_NAME) as run:
    dataset_payload = pd.concat([
        X_train.assign(target=y_train.values, split_set="train"),
        X_test.assign(target=y_test.values, split_set="test"),
    ], ignore_index=True)

    description = "RF Tuned - Current Stress; features=all; dataset=current_stress_v1; split=80/20; random_state=42; tuned via GridSearchCV"
    log_run_metadata(
        run_description=description,
        tags={"features": FEATURE_SET, "model": "RF_Tuned"},
        params={"registered_model_name": REGISTERED_MODEL_NAME, **{"model_type": "RandomForestClassifier", "tuning": "GridSearchCV"}},
        dataset_df=dataset_payload,
        dataset_context="training",
        dataset_source=dataset_source,
    )

    grid = GridSearchCV(
        estimator=model,
        param_grid={
            "classifier__n_estimators": [200, 300],
            "classifier__max_depth": [None, 8, 12],
            "classifier__min_samples_split": [2, 5],
            "classifier__class_weight": [None, "balanced"],
        },
        scoring="f1_weighted",
        cv=5,
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    mlflow.log_metric("cv_best_score", float(grid.best_score_))
    mlflow.log_params({f"best_{k}": v for k, v in grid.best_params_.items()})
    best_model.fit(X_train, y_train)

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test) if hasattr(best_model, "predict_proba") else None
    metrics = evaluate_classification(y_test, y_pred, y_proba)
    mlflow.log_metrics(metrics)

    local_output_dir = create_local_output_dir(repo_root, NOTEBOOK_NAME, run.info.run_id)
    log_classification_artifacts(y_test, y_pred, local_output_dir, y_proba, prefix="test")

    (local_output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    joblib.dump(best_model, local_output_dir / "model.joblib")

    mlflow.log_artifacts(str(local_output_dir), artifact_path="local_outputs")

    signature = infer_signature(X_train, best_model.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(5),
        registered_model_name=REGISTERED_MODEL_NAME,
    )

    print("Run ID:", run.info.run_id)
    print("Local output:", local_output_dir)
    print("Registered model:", REGISTERED_MODEL_NAME)
    print(pd.Series(metrics).sort_index())
